In [2]:
import pandas as pd
df_menu = pd.read_excel(r'C:\Users\YangTeng\Downloads\爬虫\cardone-brake caliper\menu.xlsx', header=0)
list_url = df_menu['Url'].to_list()

print('总数量：' + str(len(list_url)))
list_url

总数量：9239


In [3]:
df = pd.DataFrame(columns=['Url',
                           'Part_Number',
                           'Src',
                           'status'])

df

,Url,Part_Number,Src,status


In [5]:
import requests
i = 1

resp = requests.get(list_url[i]).text

resp

ProxyError: HTTPSConnectionPool(host='uacparts.com', port=443): Max retries exceeded with url: /Catalog/CrossReference?__aweconid=CRP6&part=CO%2020067C (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 449 Foreign Host Forbidden')))

In [14]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(resp, 'lxml')

print(soup.prettify())

<!DOCTYPE html>
<html class="no-js" lang="en">
 <head>
  <title>
   Brake Caliper - 18-4003
  </title>
  <link crossorigin="" href="https://cdn11.bigcommerce.com/s-z9vhe3o238" rel="dns-prefetch preconnect"/>
  <link crossorigin="" href="https://fonts.googleapis.com/" rel="dns-prefetch preconnect"/>
  <link crossorigin="" href="https://fonts.gstatic.com/" rel="dns-prefetch preconnect"/>
  <meta content="0" property="product:price:amount"/>
  <meta content="USD" property="product:price:currency"/>
  <meta content="https://www.cardone.com/product/unloaded-caliper-18-4003" property="og:url"/>
  <meta content="CARDONE Industries" property="og:site_name"/>
  <meta content="CARDONE Reman,Brake Caliper,Chevrolet Chevette 1979-78; Pontiac Acadian 1979-78" name="description"/>
  <link href="https://www.cardone.com/product/unloaded-caliper-18-4003" rel="canonical"/>
  <meta content="bigcommerce.stencil" name="platform"/>
  <meta content="product" property="og:type"/>
  <meta content="Brake Calipe

In [15]:
from lxml import etree

html = etree.HTML(str(soup))

part_number = html.xpath('//dd[@data-product-sku]/text()')[0]

part_number

'18-4003'

In [16]:
src = html.xpath('//div[@class="productView-img-container"]/a/@href')[0]

src

'https://cdn11.bigcommerce.com/s-z9vhe3o238/images/stencil/1280x1280/products/54530/5514290/482__86004.1687987719.jpg?c=1'

In [17]:
df_temp = pd.DataFrame([{'Url': list_url[i],
                         'Part_Number': part_number,
                         'Src': src,
                         'status': 'ok'}])

df_temp

,Url,Part_Number,Src,status
0,https://www.cardone.com/product/unloaded-calip...,18-4003,https://cdn11.bigcommerce.com/s-z9vhe3o238/ima...,ok


In [18]:
list_general = html.xpath('//div[@class="productView-table general"]/div[@class="productView-table-row"]')

list_general

[<Element div at 0x199cedba300>,
 <Element div at 0x199cefd5c00>,
 <Element div at 0x199cefd5940>,
 <Element div at 0x199cefd60c0>,
 <Element div at 0x199cefd5a80>,
 <Element div at 0x199cefd4e00>,
 <Element div at 0x199cefd4600>,
 <Element div at 0x199cefd5880>]

In [9]:
for row in list_general:
    name = row.xpath('./div[@class="productView-table-header"]/text()')[0]
    if name.endswith(':'):
        name = name[:-1]
    value = row.xpath('./div[@class="productView-table-data"]/text()')[0]
    df_temp[name.lower()] = value
    
list_technical = html.xpath('//div[@class="productView-table technical"]/div[@class="productView-table-row"]')

list_technical

[<Element div at 0x19e78172580>,
 <Element div at 0x19e78173f80>,
 <Element div at 0x19e7815c680>,
 <Element div at 0x19e7815c800>,
 <Element div at 0x19e7815c880>,
 <Element div at 0x19e7815c900>,
 <Element div at 0x19e7815c940>,
 <Element div at 0x19e7815c9c0>,
 <Element div at 0x19e7815ca40>,
 <Element div at 0x19e7815c340>,
 <Element div at 0x19e7815ca00>]

In [10]:
for row in list_technical:
    name = row.xpath('./div[@class="productView-table-header"]/text()')[0]
    if name.endswith(':'):
        name = name[:-1]
    value = row.xpath('./div[@class="productView-table-data"]/text()')[0]
    df_temp[name.lower()] = value
    
df = pd.concat([df, df_temp], ignore_index=True).fillna('')

df

,Url,Part_Number,Src,status,part type,product condition,lifecycle status,package contents,packaged weight,upc,...,color,finish,inlet port size,installation hardware included,mounting bolts included,pads included,piston material,piston quantity,piston size od - imperial,piston size od - metric
0,https://www.cardone.com/product/unloaded-calip...,18-4003,https://cdn11.bigcommerce.com/s-z9vhe3o238/ima...,ok,Brake Caliper,Remanufactured,Available to Order,Caliper; Instruction Sheet; Hardware Kit,4.4900 lb,00082617010498,...,Gray,Powder Coated,0.39,Yes,Yes,No,Steel,1,1.87 in,47.52 mm
